---
## Working with data in Pandas

In [ ]:
import re, pandas
from src.backend.tools import *
from src.backend.get_pars import *

In [ ]:
# ===============================================
# Old Function kept just because; I like the trick
def table_to_csv_str(table: list[list]) -> str :
    """
    Takes 'table' of Lists
        [L1, L2, ...]
    makes `,`-separated strings of the L_i's respective entries,
    and then concatenates these `,`-separated strings with newline `\n` separators
    """
    rows = [",".join(map(str,row)) for row in table]
    return "\n".join(rows)

---
### Using `pandas` to interface

In [ ]:
df_min = 1990
df_max = 2026

df = pd.read_csv(csv_dir/f"pars{df_min}.csv")
for file in csv_dir.glob("*.csv"):
    if df_min < int(file.stem[-4:]) <= df_max :
        _df = pd.read_csv(file)
        df = pd.concat([df, _df], ignore_index=True)

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9180 entries, 0 to 9179
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    9180 non-null   str    
 1   1M      6280 non-null   float64
 2   2M      1975 non-null   float64
 3   3M      9176 non-null   float64
 4   4M      973 non-null    float64
 5   6M      9179 non-null   float64
 6   1Y      9179 non-null   float64
 7   2Y      9179 non-null   float64
 8   3Y      9179 non-null   float64
 9   5Y      9179 non-null   float64
 10  7Y      9179 non-null   float64
 11  10Y     9179 non-null   float64
 12  20Y     8240 non-null   float64
 13  30Y     8185 non-null   float64
dtypes: float64(13), str(1)
memory usage: 1.1 MB


In [41]:
df.loc[df["1Y"].isnull()]

,date,1M,2M,3M,4M,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
6122,2010-10-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
import pandas as pd
import sqlite3 as sql



csv_file = "./misc/db_test/pars2024.csv"
db_file = "./misc/db_test/pars2020s.db"

df = pd.read_csv(csv_file)

conn = sq.connect(db_file)
df.to_sql("pars2024", conn, index=False)
conn.close()

---
### Using `sqlite3` directly

In [40]:
import csv
import sqlite3

db_file = "./misc/db_test/pars2024.csv"

with open(db_file, 'r') as file:
    headers = next(csv.reader(file))
cols = ", ".join(f'\"{h}\" REAL' if h != 'date' else '\"date\" TEXT' for h in headers)

conn = sqlite3.connect("./misc/db_test/pars2020s.db")
cur = conn.cursor()
cur.execute(f"CREATE TABLE pars2024 ({cols});")
cur.execute(f".mode csv; .import")
conn.close()

In [ ]:
import sqlite3

def main():
    conn = sqlite3.connect("db01")
    cur = conn.cursor()

    # note: cur.execute is more like an 'initialized iterator' for your query. It doesn't actually pull the data
    # until you do something like `cur.fetchall()` or `cur.fetchone()` or `cur.fetchmany()` (??).
    # this prevents something like trying to pull in a list of millions of entries all at once.
    # cur.execute("SELECT * FROM pars202411 WHERE age > ?", (18,))
    cur.execute("SELECT * FROM pars202411")

    row = cur.fetchone()
    print(type(row))
    print(type(row[0]), "\t", type(row[1]))

    cur.close()
    conn.close()